In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# loading all 4 datasets we downloaded
deliveries   = pd.read_csv(r'D:\IPL_Data_Analysis\deliveries_updated_mens_ipl_upto_2024.csv')
matches      = pd.read_csv(r'D:\IPL_Data_Analysis\matches_updated_mens_ipl_upto_2024.csv')
auction_2024 = pd.read_csv(r'D:\IPL_Data_Analysis\IPL_2024_Players_Auction_Dataset.csv')
auction_hist = pd.read_csv(r'D:\IPL_Data_Analysis\IPLPlayerAuctionData.csv')

# just checking if data loaded properly
# want to see shape, column names and how many nulls we have
datasets = {
    "Deliveries"     : deliveries,
    "Matches"        : matches,
    "Auction 2024"   : auction_2024,
    "Auction History": auction_hist
}

for name, df in datasets.items():
    print(f"\n{name}")
    print(f"rows and columns : {df.shape}")
    print(f"column names     : {list(df.columns)}")
    print(f"total null values: {df.isnull().sum().sum()}")
    print("-" * 50)

In [ ]:
# ---- Deep dive ----

# 1. Check auction 2024 nulls column-wise
print("=== Auction 2024 Null Breakdown ===")
print(auction_2024.isnull().sum())

# 2. Sample rows
print("\n=== Auction 2024 Sample ===")
print(auction_2024.head())

# 3. Auction history year range
print("\n=== Auction History Year Range ===")
print(auction_hist['Year'].value_counts().sort_index())

# 4. Unique players in deliveries
print("\n=== Unique Batsmen in Deliveries ===")
print(deliveries['batsman'].nunique())

print("\n=== Unique Bowlers in Deliveries ===")
print(deliveries['bowler'].nunique())

# 5. Seasons available
print("\n=== Seasons in Matches ===")
print(matches['season'].value_counts().sort_index())

In [ ]:
# PHASE 2 - cleaning the data before we can use it

# deliveries dataset
deliveries_clean = deliveries.drop(columns=['over_ball', 'Byes', 'LegByes', 'Penalty'])

# if no dismissal happened just fill it as not out
deliveries_clean['dismissal_kind'] = deliveries_clean['dismissal_kind'].fillna('not out')
deliveries_clean['player_dismissed'] = deliveries_clean['player_dismissed'].fillna('none')

deliveries_clean['date'] = pd.to_datetime(deliveries_clean['date'])

# removing extra spaces from names if any
deliveries_clean['batsman'] = deliveries_clean['batsman'].str.strip()
deliveries_clean['bowler']  = deliveries_clean['bowler'].str.strip()

print(f"deliveries shape: {deliveries_clean.shape}")
print(f"nulls left: {deliveries_clean.isnull().sum().sum()}")


# matches dataset - keeping only columns we actually need
matches_clean = matches[['matchId', 'season', 'date', 'team1', 'team2',
                          'winner', 'venue', 'city', 'toss_winner',
                          'toss_decision', 'player_of_match', 'outcome',
                          'winner_runs', 'winner_wickets']].copy()

matches_clean['date'] = pd.to_datetime(matches_clean['date'])

matches_clean['winner']         = matches_clean['winner'].fillna('No Result')
matches_clean['city']           = matches_clean['city'].fillna('Unknown')
matches_clean['winner_runs']    = matches_clean['winner_runs'].fillna(0)
matches_clean['winner_wickets'] = matches_clean['winner_wickets'].fillna(0)

print(f"matches shape: {matches_clean.shape}")
print(f"nulls left: {matches_clean.isnull().sum().sum()}")


# auction 2024 dataset
# unnamed column and dollar column are useless, dropping them
auction_2024_clean = auction_2024.drop(columns=['Unnamed: 0', 'Cost IN $ (000)'])

auction_2024_clean.columns = ['player', 'base_price', 'role', 'sold_price_cr', '2023_squad', 'team']

# base price was in raw numbers like 5000000, converting to crores
auction_2024_clean['base_price'] = pd.to_numeric(auction_2024_clean['base_price'], errors='coerce')
auction_2024_clean['base_price_cr'] = auction_2024_clean['base_price'] / 10_000_000
auction_2024_clean['base_price_cr'] = auction_2024_clean['base_price_cr'].fillna(0)
auction_2024_clean = auction_2024_clean.drop(columns=['base_price'])

# null sold price means the player was unsold
auction_2024_clean['sold_price_cr'] = auction_2024_clean['sold_price_cr'].fillna(0)
auction_2024_clean['status'] = auction_2024_clean['sold_price_cr'].apply(
    lambda x: 'sold' if x > 0 else 'unsold'
)

auction_2024_clean['player'] = auction_2024_clean['player'].str.strip()
auction_2024_clean['role']   = auction_2024_clean['role'].str.strip()

print(f"auction 2024 shape: {auction_2024_clean.shape}")
print(f"sold: {(auction_2024_clean['status'] == 'sold').sum()}")
print(f"unsold: {(auction_2024_clean['status'] == 'unsold').sum()}")


# historical auction data
auction_hist_clean = auction_hist.copy()

auction_hist_clean.columns = ['player', 'role', 'amount_cr', 'team', 'year', 'origin']

auction_hist_clean['origin'] = auction_hist_clean['origin'].fillna('Unknown')
auction_hist_clean['player'] = auction_hist_clean['player'].str.strip()
auction_hist_clean['role']   = auction_hist_clean['role'].str.strip()

print(f"auction history shape: {auction_hist_clean.shape}")
print(f"nulls left: {auction_hist_clean.isnull().sum().sum()}")


# saving all cleaned files
deliveries_clean.to_csv(r'D:\IPL_Data_Analysis\data\processed\deliveries_clean.csv', index=False)
matches_clean.to_csv(r'D:\IPL_Data_Analysis\data\processed\matches_clean.csv', index=False)
auction_2024_clean.to_csv(r'D:\IPL_Data_Analysis\data\processed\auction_2024_clean.csv', index=False)
auction_hist_clean.to_csv(r'D:\IPL_Data_Analysis\data\processed\auction_hist_clean.csv', index=False)

print("all cleaned files saved")

In [ ]:
# PHASE 3 - building features from raw data
# idea is to treat each player like a stock and calculate their value

deliveries_clean = pd.read_csv(r'D:\IPL_Data_Analysis\data\processed\deliveries_clean.csv')
matches_clean    = pd.read_csv(r'D:\IPL_Data_Analysis\data\processed\matches_clean.csv')
auction_2024     = pd.read_csv(r'D:\IPL_Data_Analysis\data\processed\auction_2024_clean.csv')
auction_hist     = pd.read_csv(r'D:\IPL_Data_Analysis\data\processed\auction_hist_clean.csv')


# batting stats per player
batting = deliveries_clean.groupby('batsman').agg(
    total_runs    = ('batsman_runs', 'sum'),
    total_balls   = ('batsman_runs', 'count'),
    innings_count = ('matchId', 'nunique'),
    fours         = ('batsman_runs', lambda x: (x == 4).sum()),
    sixes         = ('batsman_runs', lambda x: (x == 6).sum()),
).reset_index()

batting['strike_rate'] = (batting['total_runs'] / batting['total_balls']) * 100
batting['batting_avg'] = batting['total_runs'] / batting['innings_count']
batting['boundary_pct'] = ((batting['fours'] + batting['sixes']) / batting['total_balls']) * 100

# consistency score - higher avg and higher strike rate = more valuable player
batting['batting_consistency_score'] = (batting['batting_avg'] * batting['strike_rate']) / 100

print(f"batting features done for {len(batting)} players")


# bowling stats per player
bowling = deliveries_clean.groupby('bowler').agg(
    wickets        = ('dismissal_kind', lambda x: (x != 'not out').sum()),
    runs_conceded  = ('batsman_runs', 'sum'),
    balls_bowled   = ('batsman_runs', 'count'),
    innings_bowled = ('matchId', 'nunique'),
).reset_index()

bowling['economy_rate'] = (bowling['runs_conceded'] / bowling['balls_bowled']) * 6

# if wickets = 0 just put 999 so it doesn't break
bowling['bowling_avg'] = bowling.apply(
    lambda x: x['runs_conceded'] / x['wickets'] if x['wickets'] > 0 else 999, axis=1
)
bowling['bowling_sr'] = bowling.apply(
    lambda x: x['balls_bowled'] / x['wickets'] if x['wickets'] > 0 else 999, axis=1
)

# higher score = better bowler
bowling['bowling_value_score'] = bowling.apply(
    lambda x: (x['wickets'] * 100) / (x['economy_rate'] * x['bowling_sr'])
    if x['wickets'] > 0 and x['economy_rate'] > 0 and x['bowling_sr'] > 0 else 0, axis=1
)

print(f"bowling features done for {len(bowling)} players")


# merging batting and bowling together
# using outer join so all-rounders keep both stats
batting_df = batting.rename(columns={'batsman': 'player'})
bowling_df = bowling.rename(columns={'bowler': 'player'})

player_stats = pd.merge(batting_df, bowling_df, on='player', how='outer')
player_stats = player_stats.fillna(0)

print(f"combined player stats: {player_stats.shape}")


# value index - this is the main thing that makes this project unique
# treating players like stocks and scoring them 0 to 100
def normalize(series):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return series * 0
    return ((series - min_val) / (max_val - min_val)) * 100

player_stats['batting_score_norm'] = normalize(player_stats['batting_consistency_score'])
player_stats['bowling_score_norm'] = normalize(player_stats['bowling_value_score'])

# t20 is more batting heavy so giving batting 60% weight
player_stats['value_index'] = (
    player_stats['batting_score_norm'] * 0.6 +
    player_stats['bowling_score_norm'] * 0.4
)

print(f"value index calculated for {len(player_stats)} players")


# merging with auction data to get the final dataset
final_df = pd.merge(auction_2024, player_stats, on='player', how='left')
final_df = final_df.fillna(0)

final_df_sold = final_df[final_df['status'] == 'sold'].copy()

final_df.to_csv(r'D:\IPL_Data_Analysis\data\processed\final_player_features.csv', index=False)
final_df_sold.to_csv(r'D:\IPL_Data_Analysis\data\processed\final_sold_players.csv', index=False)

print(f"final dataset: {final_df.shape}")
print(f"sold players only: {final_df_sold.shape}")

print("\ntop 10 players by value index:")
print(player_stats.nlargest(10, 'value_index')[
    ['player', 'value_index', 'batting_consistency_score', 'bowling_value_score']
].to_string(index=False))

In [ ]:
# PHASE 4 - SQL analysis
# answering business questions like which team is the smartest investor

import sqlite3

conn = sqlite3.connect(':memory:')

deliveries_clean.to_sql('deliveries', conn, index=False, if_exists='replace')
matches_clean.to_sql('matches', conn, index=False, if_exists='replace')
final_df.to_sql('players', conn, index=False, if_exists='replace')
auction_hist.to_sql('auction_hist', conn, index=False, if_exists='replace')

print("tables loaded into sqlite")


# which franchise got the most value per crore spent
q1 = pd.read_sql_query("""
    SELECT 
        team,
        COUNT(player)                          AS players_bought,
        ROUND(SUM(sold_price_cr), 2)           AS total_spent_cr,
        ROUND(AVG(sold_price_cr), 2)           AS avg_price_cr,
        ROUND(MAX(sold_price_cr), 2)           AS max_spend_cr,
        ROUND(AVG(value_index), 2)             AS avg_value_index,
        ROUND(SUM(value_index) / 
              SUM(sold_price_cr), 2)           AS roi_per_crore
    FROM players
    WHERE status = 'sold' AND sold_price_cr > 0
    GROUP BY team
    ORDER BY roi_per_crore DESC
""", conn)

print("\nfranchise investment analysis:")
print(q1.to_string(index=False))


# finding overvalued and undervalued players - the stock screener
q2 = pd.read_sql_query("""
    SELECT
        player,
        role,
        team,
        sold_price_cr,
        ROUND(value_index, 2) AS value_index,
        ROUND(sold_price_cr / NULLIF(value_index, 0), 4) AS price_per_value,
        CASE 
            WHEN value_index > 50 AND sold_price_cr < 5 THEN 'UNDERVALUED'
            WHEN value_index < 20 AND sold_price_cr > 8 THEN 'OVERVALUED'
            ELSE 'FAIRLY PRICED'
        END AS market_status
    FROM players
    WHERE status = 'sold'
    ORDER BY value_index DESC
    LIMIT 20
""", conn)

print("\novervalued vs undervalued players:")
print(q2.to_string(index=False))


# best value player in each role
q3 = pd.read_sql_query("""
    SELECT
        role,
        player,
        team,
        sold_price_cr,
        ROUND(value_index, 2) AS value_index,
        ROUND(total_runs, 0)  AS total_runs,
        ROUND(strike_rate, 2) AS strike_rate,
        ROUND(wickets, 0)     AS wickets
    FROM players
    WHERE status = 'sold'
    AND value_index = (
        SELECT MAX(p2.value_index) 
        FROM players p2 
        WHERE p2.role = players.role 
        AND p2.status = 'sold'
    )
    ORDER BY value_index DESC
""", conn)

print("\nbest value player per role:")
print(q3.to_string(index=False))


# price history of top players across years - like a stock chart
q4 = pd.read_sql_query("""
    SELECT
        player,
        year,
        ROUND(amount_cr, 2) AS amount_cr,
        team
    FROM auction_hist
    WHERE player IN (
        SELECT player FROM auction_hist
        GROUP BY player
        HAVING COUNT(*) >= 3
        ORDER BY MAX(amount_cr) DESC
        LIMIT 8
    )
    ORDER BY player, year
""", conn)

print("\nhistorical price trend:")
print(q4.to_string(index=False))


# which team has the strongest batting lineup
q5 = pd.read_sql_query("""
    SELECT
        team,
        ROUND(AVG(batting_avg), 2)               AS avg_batting_avg,
        ROUND(AVG(strike_rate), 2)               AS avg_strike_rate,
        ROUND(AVG(batting_consistency_score), 2) AS avg_consistency,
        COUNT(player)                            AS num_batters
    FROM players
    WHERE status = 'sold'
    AND total_runs > 100
    GROUP BY team
    ORDER BY avg_consistency DESC
""", conn)

print("\nteam batting strength:")
print(q5.to_string(index=False))


q1.to_csv(r'D:\IPL_Data_Analysis\data\processed\franchise_roi.csv', index=False)
q2.to_csv(r'D:\IPL_Data_Analysis\data\processed\player_valuation.csv', index=False)

print("query results saved")

In [ ]:
# PHASE 5 - ML model to predict fair auction price
# using auction history (970 rows) instead of just 2024 data
# more data = better model

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

ml_df = auction_hist_clean.copy()

# converting to crores if it's in raw numbers
if ml_df['amount_cr'].max() > 1000:
    ml_df['amount_cr'] = ml_df['amount_cr'] / 10_000_000

# encoding text columns to numbers since ml models need numbers
le_role   = LabelEncoder()
le_origin = LabelEncoder()
le_team   = LabelEncoder()

ml_df['role_enc']   = le_role.fit_transform(ml_df['role'].astype(str))
ml_df['origin_enc'] = le_origin.fit_transform(ml_df['origin'].astype(str))
ml_df['team_enc']   = le_team.fit_transform(ml_df['team'].astype(str))

# adding actual performance stats to make predictions better
ml_df = ml_df.merge(
    player_stats[['player', 'total_runs', 'strike_rate',
                  'batting_avg', 'wickets', 'economy_rate', 'value_index']],
    on='player', how='left'
).fillna(0)

features = ['role_enc', 'origin_enc', 'team_enc', 'year',
            'total_runs', 'strike_rate', 'batting_avg',
            'wickets', 'economy_rate', 'value_index']

X = ml_df[features].fillna(0)
y = ml_df['amount_cr'].fillna(0)

print(f"features: {X.shape[1]}, samples: {X.shape[0]}")
print(f"price range: {y.min():.2f} Cr to {y.max():.2f} Cr")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# trying 3 different models and picking the best one
models = {
    'Random Forest'     : RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting' : GradientBoostingRegressor(n_estimators=200, random_state=42),
    'XGBoost'           : xgb.XGBRegressor(n_estimators=200, random_state=42, verbosity=0)
}

print(f"\n{'model':<25} {'mae (cr)':>10} {'r2 score':>10} {'cv avg':>10}")
print("-" * 58)

best_model_name = None
best_r2         = -999
results         = {}

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae   = mean_absolute_error(y_test, preds)
    r2    = r2_score(y_test, preds)
    results[name] = {'model': model, 'mae': mae, 'r2': r2}
    print(f"{name:<25} {mae:>10.2f} {r2:>10.4f} {cv_scores.mean():>10.4f}")
    if r2 > best_r2:
        best_r2         = r2
        best_model_name = name

print(f"\nbest model: {best_model_name}")

best_model = results[best_model_name]['model']

# which features matter the most for price prediction
importance_df = pd.DataFrame({
    'feature'   : features,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nfeature importance:")
print(importance_df.to_string(index=False))


# predicting fair price for all 2024 auction players
predict_df = auction_2024_clean.copy()

predict_df['role_enc'] = predict_df['role'].apply(
    lambda x: le_role.transform([x])[0] if x in le_role.classes_ else 0
)
predict_df['origin_enc'] = 0
predict_df['team_enc'] = predict_df['team'].apply(
    lambda x: le_team.transform([x])[0] if x in le_team.classes_ else 0
)
predict_df['year'] = 2024

predict_df = predict_df.merge(
    player_stats[['player', 'total_runs', 'strike_rate',
                  'batting_avg', 'wickets', 'economy_rate', 'value_index']],
    on='player', how='left'
).fillna(0)

predict_df['predicted_price_cr'] = best_model.predict(predict_df[features]).round(2)
predict_df['predicted_price_cr'] = predict_df['predicted_price_cr'].clip(lower=0)

# value gap tells us if a player was bought at fair price or not
predict_df['value_gap'] = (predict_df['sold_price_cr'] - predict_df['predicted_price_cr']).round(2)
predict_df['valuation'] = predict_df['value_gap'].apply(
    lambda x: 'OVERVALUED' if x > 2 else ('UNDERVALUED' if x < -2 else 'FAIR')
)

sold = predict_df[predict_df['status'] == 'sold']

print("\nmost undervalued players (hidden gems):")
print(sold.nsmallest(5, 'value_gap')[
    ['player', 'role', 'team', 'sold_price_cr', 'predicted_price_cr', 'value_gap', 'valuation']
].to_string(index=False))

print("\nmost overvalued players:")
print(sold.nlargest(5, 'value_gap')[
    ['player', 'role', 'team', 'sold_price_cr', 'predicted_price_cr', 'value_gap', 'valuation']
].to_string(index=False))

predict_df.to_csv(r'D:\IPL_Data_Analysis\data\processed\final_with_predictions.csv', index=False)
print("predictions saved")

In [ ]:
# PHASE 6 - visualizations
# making the analysis presentable with charts

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# dark theme - looks cleaner for data viz
plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor']   = '#161b22'
plt.rcParams['axes.edgecolor']   = '#30363d'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'
plt.rcParams['grid.color']       = '#21262d'
plt.rcParams['grid.alpha']       = 0.5
plt.rcParams['font.family']      = 'monospace'


# chart 1 - which franchise is the smartest investor
franchise_roi = pd.read_csv(r'D:\IPL_Data_Analysis\data\processed\franchise_roi.csv')

fig, ax = plt.subplots(figsize=(14, 7))

colors = ['#00ff88' if x == franchise_roi['roi_per_crore'].max()
          else '#ff4444' if x == franchise_roi['roi_per_crore'].min()
          else '#4dabf7'
          for x in franchise_roi['roi_per_crore']]

bars = ax.barh(franchise_roi['team'], franchise_roi['roi_per_crore'],
               color=colors, edgecolor='none', height=0.6)

for bar, val in zip(bars, franchise_roi['roi_per_crore']):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=11, color='white')

ax.set_xlabel('RoI per Crore Spent', fontsize=13)
ax.set_title('IPL 2024 — Franchise Investment RoI\n(value index earned per 1 crore spent)',
             fontsize=15, pad=20)
ax.axvline(x=franchise_roi['roi_per_crore'].mean(), color='yellow',
           linestyle='--', alpha=0.7)
ax.grid(axis='x')

green_patch = mpatches.Patch(color='#00ff88', label='best investor')
red_patch   = mpatches.Patch(color='#ff4444', label='worst investor')
blue_patch  = mpatches.Patch(color='#4dabf7', label='other teams')
ax.legend(handles=[green_patch, red_patch, blue_patch], fontsize=10)

plt.tight_layout()
plt.savefig(r'D:\IPL_Data_Analysis\chart1_franchise_roi.png', dpi=150, bbox_inches='tight')
plt.show()


# chart 2 - top 20 players by value index (like a stock screener)
top_players = player_stats[player_stats['innings_count'] >= 10].nlargest(20, 'value_index')

fig, ax = plt.subplots(figsize=(14, 8))

colors = plt.cm.RdYlGn(
    [x / top_players['value_index'].max() for x in top_players['value_index']]
)

bars = ax.barh(top_players['player'], top_players['value_index'],
               color=colors, edgecolor='none', height=0.65)

for bar, val in zip(bars, top_players['value_index']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=10, color='white')

ax.set_xlabel('value index score', fontsize=13)
ax.set_title('IPL player value index — top 20\n(like a stock screener for cricket players)',
             fontsize=15, pad=20)
ax.grid(axis='x')

plt.tight_layout()
plt.savefig(r'D:\IPL_Data_Analysis\chart2_player_value_index.png', dpi=150, bbox_inches='tight')
plt.show()


# chart 3 - actual price vs what our model predicted
sold_predictions = predict_df[predict_df['status'] == 'sold'].copy()

fig, ax = plt.subplots(figsize=(12, 8))

color_map = {
    'OVERVALUED'  : '#ff4444',
    'FAIR'        : '#ffd700',
    'UNDERVALUED' : '#00ff88'
}
point_colors = sold_predictions['valuation'].map(color_map).fillna('#4dabf7')

ax.scatter(
    sold_predictions['predicted_price_cr'],
    sold_predictions['sold_price_cr'],
    c=point_colors, s=120, alpha=0.85, edgecolors='white', linewidth=0.5
)

# if a point is on this line, model predicted perfectly
max_val = max(sold_predictions['sold_price_cr'].max(),
              sold_predictions['predicted_price_cr'].max())
ax.plot([0, max_val], [0, max_val], color='white',
        linestyle='--', alpha=0.5, linewidth=1.5, label='perfect prediction')

for _, row in sold_predictions.nlargest(5, 'sold_price_cr').iterrows():
    ax.annotate(row['player'],
                xy=(row['predicted_price_cr'], row['sold_price_cr']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=8, color='white', alpha=0.9)

ax.set_xlabel('predicted price (cr)', fontsize=13)
ax.set_ylabel('actual auction price (cr)', fontsize=13)
ax.set_title('actual vs predicted auction price\n(above the line = overvalued, below = undervalued)',
             fontsize=14, pad=20)

patches = [mpatches.Patch(color=v, label=k.lower()) for k, v in color_map.items()]
ax.legend(handles=patches, fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(r'D:\IPL_Data_Analysis\chart3_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()


# chart 4 - player price history over the years, exactly like a stock chart
top_hist_players = auction_hist_clean.groupby('player')['amount_cr'].max()\
                   .nlargest(6).index.tolist()

hist_filtered = auction_hist_clean[
    auction_hist_clean['player'].isin(top_hist_players)
].copy()

if hist_filtered['amount_cr'].max() > 1000:
    hist_filtered['amount_cr'] = hist_filtered['amount_cr'] / 10_000_000

fig, ax = plt.subplots(figsize=(14, 7))

colors_line = ['#00ff88', '#4dabf7', '#ffd700', '#ff4444', '#ff8c00', '#da70d6']

for i, player in enumerate(top_hist_players):
    player_data = hist_filtered[hist_filtered['player'] == player].sort_values('year')
    ax.plot(player_data['year'], player_data['amount_cr'],
            marker='o', linewidth=2.5, markersize=7,
            color=colors_line[i], label=player)
    last = player_data.iloc[-1]
    ax.annotate(f"₹{last['amount_cr']:.1f}Cr",
                xy=(last['year'], last['amount_cr']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=8, color=colors_line[i])

ax.set_xlabel('year', fontsize=13)
ax.set_ylabel('auction price (cr)', fontsize=13)
ax.set_title('IPL player price history — like a stock chart\n(top 6 players by peak auction value)',
             fontsize=14, pad=20)
ax.legend(fontsize=10, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(r'D:\IPL_Data_Analysis\chart4_price_history.png', dpi=150, bbox_inches='tight')
plt.show()


# chart 5 - breakdown of players by role and how much each role costs on average
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

role_counts = auction_2024_clean[auction_2024_clean['status'] == 'sold']\
              .groupby('role')['player'].count()

colors_pie = ['#4dabf7', '#00ff88', '#ffd700', '#ff4444']
axes[0].pie(role_counts, labels=role_counts.index, autopct='%1.1f%%',
            colors=colors_pie, startangle=90,
            textprops={'color': 'white', 'fontsize': 12})
axes[0].set_title('players sold by role', fontsize=13, pad=15)
axes[0].set_facecolor('#161b22')

role_spend = auction_2024_clean[auction_2024_clean['status'] == 'sold']\
             .groupby('role')['sold_price_cr'].mean().sort_values(ascending=False)

bars = axes[1].bar(role_spend.index, role_spend.values,
                   color=colors_pie, edgecolor='none', width=0.5)

for bar, val in zip(bars, role_spend.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'₹{val:.1f}Cr', ha='center', fontsize=11, color='white')

axes[1].set_ylabel('avg auction price (cr)', fontsize=12)
axes[1].set_title('avg spend per role', fontsize=13, pad=15)
axes[1].grid(axis='y', alpha=0.3)

fig.suptitle('IPL 2024 auction — role analysis', fontsize=15, y=1.02)

plt.tight_layout()
plt.savefig(r'D:\IPL_Data_Analysis\chart5_role_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("all charts saved")